# Brain Tumor Classification with InceptionV3 (GoogleNet-style workflow)

This beginner-friendly notebook covers:
1. Validate image files and explore classes.
2. Split images into **70% train, 15% validation, 15% test**.
3. Load images with `image_dataset_from_directory`.
4. Apply **training-only** flip, noise, and light blur augmentation.
5. Train a pretrained **InceptionV3** model as a feature extractor.
6. Evaluate accuracy, precision, recall, F1-score, and confusion matrix.
7. Fine-tune the last layers and compare results.
8. Save the best model.

> **Important:** InceptionV3 is not the original GoogLeNet architecture, but it is a commonly available related Inception model in TensorFlow/Keras. This notebook uses InceptionV3 because that is the model specified in the brief.
>
> The model is for an educational project, not for medical diagnosis.

## 1. Install dependencies

Run this cell if the libraries are not installed in your environment. Restart the kernel if Jupyter asks you to.

In [ ]:
%pip install -q tensorflow scikit-learn pandas matplotlib seaborn pillow

## 2. Imports and configuration

**Change `DATA_DIR`** so it points to your original dataset folder. The folder should contain one subfolder per class, for example `glioma/`, `meningioma/`, `notumor/`, and `pituitary/`.

In [ ]:
from pathlib import Path
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, UnidentifiedImageError

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

cwd = Path.cwd()
root = cwd.parent
data_dir = root / "data"

DATA_DIR = data_dir / "Data"

SPLIT_DIR = data_dir / "Classification"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

print("TensorFlow version:", tf.__version__)
print("Dataset path exists:", DATA_DIR.exists())
print("Dataset path:", DATA_DIR.resolve())

## 3. Check image files and integrity

This scans each class folder, accepts common image extensions, and attempts to open each image. Files that cannot be opened are reported and excluded from the split. It does not delete anything from the original dataset.

In [ ]:
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"DATA_DIR was not found: {DATA_DIR}\n"
        "Edit DATA_DIR in Section 2 to your dataset location."
    )

class_folders = sorted(
    p for p in DATA_DIR.iterdir()
    if p.is_dir() and not p.name.startswith(".")
)
if not class_folders:
    raise ValueError("No class subfolders found inside DATA_DIR.")

valid_records = []
invalid_files = []
unsupported_files = []

for class_folder in class_folders:
    for path in class_folder.rglob("*"):
        if not path.is_file():
            continue
        if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            unsupported_files.append(str(path))
            continue
        try:
            # verify() checks file integrity; reopen to read dimensions
            with Image.open(path) as img:
                img.verify()
            with Image.open(path) as img:
                width, height = img.size
                mode = img.mode
            if width < 1 or height < 1:
                raise ValueError("Image has invalid dimensions")
            valid_records.append({
                "path": path,
                "class_name": class_folder.name,
                "width": width,
                "height": height,
                "mode": mode
            })
        except (OSError, ValueError, UnidentifiedImageError) as exc:
            invalid_files.append({"path": str(path), "error": str(exc)})

df = pd.DataFrame(valid_records)
print(f"Valid images: {len(df)}")
print(f"Invalid/corrupt supported files: {len(invalid_files)}")
print(f"Files with unsupported extensions (ignored): {len(unsupported_files)}")
print("\nValid images per class:")
display(df["class_name"].value_counts().sort_index().to_frame("image_count"))

if invalid_files:
    display(pd.DataFrame(invalid_files).head(20))
if len(df) == 0:
    raise ValueError("No valid images were found. Check DATA_DIR and file extensions.")

## 4. Explore class balance and sample images

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="class_name", order=sorted(df["class_name"].unique()))
plt.title("Number of valid images per class")
plt.xlabel("Class")
plt.ylabel("Image count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

class_names = sorted(df["class_name"].unique())
fig, axes = plt.subplots(1, len(class_names), figsize=(4 * len(class_names), 4))
if len(class_names) == 1:
    axes = [axes]

for ax, class_name in zip(axes, class_names):
    sample_path = df.loc[df["class_name"] == class_name, "path"].iloc[0]
    with Image.open(sample_path) as img:
        ax.imshow(img.convert("RGB"))
    ax.set_title(class_name)
    ax.axis("off")
plt.tight_layout()
plt.show()

display(df[["width", "height"]].describe())
display(df.groupby("mode").size().to_frame("image_count"))

## 5. Split into train, validation, and test folders

The split is stratified so each set should keep approximately the same class proportions. The split folder is recreated each time this cell runs. Your original images are not moved or changed.

If a class has too few images for stratified splitting, inspect the error and collect more examples for that class.

In [ ]:
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["class_name"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["class_name"]
)

split_frames = {"train": train_df, "validation": val_df, "test": test_df}

# Copy images to the directory structure expected by image_dataset_from_directory.
# Prefix filenames with a counter to avoid collisions from duplicate filenames.
for split_name, split_frame in split_frames.items():
    for class_name in class_names:
        (SPLIT_DIR / split_name / class_name).mkdir(parents=True, exist_ok=True)

    for idx, row in enumerate(split_frame.itertuples(index=False)):
        source_path = row.path
        destination = SPLIT_DIR / split_name / row.class_name / f"{idx:06d}_{source_path.name}"
        shutil.copy2(source_path, destination)

split_counts = pd.DataFrame({
    split_name: frame["class_name"].value_counts()
    for split_name, frame in split_frames.items()
}).fillna(0).astype(int).reindex(class_names)

display(split_counts)
print("Total images by split:")
print({name: len(frame) for name, frame in split_frames.items()})
print("Overall total:", sum(len(frame) for frame in split_frames.values()))

## 6. Load datasets with `image_dataset_from_directory`

We load all images at the same size. Shuffling is enabled for training, but disabled for validation and test so predictions can be compared reliably with labels.

In [ ]:
train_raw = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DIR / "train",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=SEED
)

val_raw = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DIR / "validation",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

test_raw = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DIR / "test",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

class_names = train_raw.class_names
num_classes = len(class_names)
print("Classes in model order:", class_names)
print("Number of classes:", num_classes)

## 7. Training-only augmentation and InceptionV3 preprocessing

InceptionV3 expects pixel values scaled to the range it was trained with. We use its `preprocess_input` function.

Augmentation is applied **only to training batches**:
- random horizontal flip,
- small random noise,
- occasional light blur using average pooling.

Validation and test images are only preprocessed, not augmented. For brain MRI, flips may not always be medically appropriate; confirm that augmentation choices are acceptable for your assignment and dataset.

In [ ]:
def prepare_validation_or_test(images, labels):
    images = tf.keras.applications.inception_v3.preprocess_input(images)
    return images, labels

def augment_and_prepare(images, labels):
    images = tf.cast(images, tf.float32)

    # Random horizontal flip
    images = tf.image.random_flip_left_right(images)

    # Add a small amount of Gaussian noise
    noise = tf.random.normal(tf.shape(images), mean=0.0, stddev=5.0)
    images = images + noise

    # Apply a light blur to some images (probability 0.25)
    def blur_image(image):
        blurred = tf.nn.avg_pool2d(
            image[tf.newaxis, ...], ksize=3, strides=1, padding="SAME"
        )[0]
        return blurred

    should_blur = tf.random.uniform(()) < 0.25
    images = tf.cond(
        should_blur,
        lambda: tf.map_fn(blur_image, images),
        lambda: images
    )

    # Keep values within the usual image range before model preprocessing
    images = tf.clip_by_value(images, 0.0, 255.0)
    images = tf.keras.applications.inception_v3.preprocess_input(images)
    return images, labels

train_ds = train_raw.map(augment_and_prepare, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_raw.map(prepare_validation_or_test, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds = test_raw.map(prepare_validation_or_test, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# Display a batch shape to confirm loading
for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Pixel range after preprocessing:", float(tf.reduce_min(images)), float(tf.reduce_max(images)))

## 8. Build the feature-extraction model

The pretrained InceptionV3 base is frozen, so its weights do not change during this first stage. We add a small classification head for the dataset's classes and train only that head.

In [ ]:
base_model = tf.keras.applications.InceptionV3(
    weights="imagenet",
    include_top=False,
    input_shape=(*IMAGE_SIZE, 3)
)
base_model.trainable = False

inputs = keras.Input(shape=(*IMAGE_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="brain_tumor_inceptionv3")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## 9. Train the classification head

We use `ModelCheckpoint` to save the best validation model and `EarlyStopping` to stop if validation loss stops improving.

In [ ]:
Path("models").mkdir(exist_ok=True)

feature_checkpoint = "models/inceptionv3_feature_extraction_best.keras"
callbacks = [
    keras.callbacks.ModelCheckpoint(
        feature_checkpoint,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    )
]

EPOCHS_FEATURE = 10  # Increase if needed and hardware allows
history_feature = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FEATURE,
    callbacks=callbacks
)

## 10. Plot training and validation performance

In [ ]:
def plot_history(history, title):
    history_df = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history_df["accuracy"], label="Train")
    axes[0].plot(history_df["val_accuracy"], label="Validation")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history_df["loss"], label="Train")
    axes[1].plot(history_df["val_loss"], label="Validation")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_history(history_feature, "Feature extraction")

## 11. Evaluation helper: accuracy, precision, recall, F1, and confusion matrix

Macro averages give each class equal weight, which is useful if classes are imbalanced. We evaluate on the test set only after training decisions for this stage are complete.

In [ ]:
def evaluate_model(model, dataset, class_names, title):
    y_true = np.concatenate([labels.numpy() for _, labels in dataset])
    probabilities = model.predict(dataset, verbose=0)
    y_pred = np.argmax(probabilities, axis=1)

    print(title)
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision (macro): {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"Recall (macro):    {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"F1-score (macro):  {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names
    )
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.title(f"Confusion matrix — {title}")
    plt.tight_layout()
    plt.show()

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }

feature_test_metrics = evaluate_model(
    model, test_ds, class_names, "Feature extraction — test set"
)

## 12. Compare training and test performance

A large gap between training and test performance can indicate overfitting. The training dataset is augmented, so its metrics are measured on augmented training images and may not be directly comparable to the unaugmented test set.

In [ ]:
def get_predictions_and_labels(model, dataset):
    y_true = np.concatenate([labels.numpy() for _, labels in dataset])
    y_pred = np.argmax(model.predict(dataset, verbose=0), axis=1)
    return y_true, y_pred

train_y_true, train_y_pred = get_predictions_and_labels(model, train_ds)
test_y_true, test_y_pred = get_predictions_and_labels(model, test_ds)

comparison_feature = pd.DataFrame([
    {
        "set": "Training (augmented)",
        "accuracy": accuracy_score(train_y_true, train_y_pred),
        "precision_macro": precision_score(train_y_true, train_y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(train_y_true, train_y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(train_y_true, train_y_pred, average="macro", zero_division=0)
    },
    {
        "set": "Test",
        "accuracy": accuracy_score(test_y_true, test_y_pred),
        "precision_macro": precision_score(test_y_true, test_y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(test_y_true, test_y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(test_y_true, test_y_pred, average="macro", zero_division=0)
    }
])
display(comparison_feature)

## 13. Fine-tune the last layers of InceptionV3

Now unfreeze only a small part of the pretrained base. We recompile with a **much smaller learning rate** so pretrained weights change gently. Batch-normalization layers remain frozen for stability.

In [ ]:
# Start from the best saved feature-extraction model
model = keras.models.load_model(feature_checkpoint)
base_model = model.get_layer("inception_v3")

base_model.trainable = True

# Freeze most layers; fine-tune only the last 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers[-30:]:
    # Keep batch normalization layers frozen
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 14. Train the fine-tuned model

In [ ]:
fine_tuned_checkpoint = "models/inceptionv3_fine_tuned_best.keras"
fine_tune_callbacks = [
    keras.callbacks.ModelCheckpoint(
        fine_tuned_checkpoint,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    )
]

EPOCHS_FINE_TUNE = 8
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE_TUNE,
    callbacks=fine_tune_callbacks
)

plot_history(history_fine, "Fine-tuning")

## 15. Evaluate fine-tuned model and compare

Use the same test set for both stages so the comparison is fair. Do not repeatedly tune model choices based on test results; use validation performance for decisions.

In [ ]:
fine_tuned_model = keras.models.load_model(fine_tuned_checkpoint)

fine_test_metrics = evaluate_model(
    fine_tuned_model, test_ds, class_names, "Fine-tuning — test set"
)

comparison = pd.DataFrame([
    {"stage": "Feature extraction", **feature_test_metrics},
    {"stage": "Fine-tuning", **fine_test_metrics}
])
display(comparison)

comparison.set_index("stage")[["accuracy", "precision_macro", "recall_macro", "f1_macro"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1), rot=0
)
plt.title("Test-set comparison: feature extraction vs fine-tuning")
plt.ylabel("Score")
plt.tight_layout()
plt.show()

## 16. Save the final model and class names

The `.keras` file contains the model architecture and weights. Save the class names too, so predicted class indexes can be interpreted correctly later.

In [ ]:
final_model_path = "models/brain_tumor_inceptionv3_final.keras"
fine_tuned_model.save(final_model_path)

class_names_path = Path("models/class_names.txt")
class_names_path.write_text("\n".join(class_names), encoding="utf-8")

print("Saved model:", final_model_path)
print("Saved class names:", class_names_path)

## 17. Predict one image (optional)

Change the path to a single image. The preprocessing must match the training pipeline, but **do not augment** a prediction image.

In [ ]:
def predict_image(image_path, model_path=final_model_path):
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    try:
        with Image.open(image_path) as img:
            img = img.convert("RGB").resize(IMAGE_SIZE)
            image_array = np.asarray(img, dtype=np.float32)
    except (OSError, UnidentifiedImageError) as exc:
        raise ValueError(f"Could not read a valid image: {image_path}") from exc

    image_array = tf.keras.applications.inception_v3.preprocess_input(image_array)
    image_array = np.expand_dims(image_array, axis=0)

    loaded_model = keras.models.load_model(model_path)
    probabilities = loaded_model.predict(image_array, verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))

    return {
        "predicted_class": class_names[predicted_index],
        "confidence": float(probabilities[predicted_index])
    }

# Example:
# predict_image("/path/to/an/image.jpg")